In [8]:
!pip install pandas numpy
!pip install --upgrade pip setuptools wheel
!pip install "Cython<3" "numpy<2"
!pip install scikit-surprise

In [ ]:
import json
import pandas as pd

from surprise import SVD
from surprise import Dataset
from surprise import accuracy
from surprise.model_selection.split import train_test_split

from collections import defaultdict

In [10]:
with open("config.json", "r") as f:
    CONFIG = json.load(f)

In [11]:
requests = {
    "Recommend movies like Nightmare on Elm Street, Friday the 13th, and Halloween.":
    "Adequate recommendations will be horror movies from the 1970s or 1980s or sequels to the movies.",

    "Given that I like The Lion King, Pocahontas, and The Beauty and the Beast, can you recommend some movies?":
    "Adequate recommendations will be (2-D) animated movies or real-life remakes of Disney movies.",

    "Recommend movies similar to Hamlet and Othello.":
    "Adequate recommendations will be movies in the drama genre that are based on classic literature (e.g., Shakespeare, Dickens, or Jane Austen).",    
}

In [12]:
movie_df = pd.read_csv('cache/recommendation/movies.csv', usecols=['movieId', 'title'])
movie_df.head()

,movieId,title
0,1,Toy Story (1995)
1,2,Jumanji (1995)
2,3,Grumpier Old Men (1995)
3,4,Waiting to Exhale (1995)
4,5,Father of the Bride Part II (1995)


## Factorization Methods

While it is also possible to implement a factorization approach using sklearn, be using the [Surprise](http://surpriselib.com/) library for this part of the tutorial to show you an out-of-the-box approach. Surprise is one of many frameworks that provide a selection of recommendation methods for explicit feedback data in a ready-to-use manner. It was heavily inspired by sklearn and uses a similar syntax and provides builtin datasets and evaluation metrics.

We will compare the performance of 2 factorization methods, Singular Value Decomposition (SVD) which became popular when it lead to winning the third place in the Netflix competition, and the similar Non-Negative Matrix Factorization (NMF). You can check out the maths behind both approaches in the documentation ([Link to SVD](https://surprise.readthedocs.io/en/stable/matrix_factorization.html#surprise.prediction_algorithms.matrix_factorization.SVD), [Link to NMF](https://surprise.readthedocs.io/en/stable/matrix_factorization.html#surprise.prediction_algorithms.matrix_factorization.NMF)).

Surprise uses their own data format. They provide functions to load external data from a file or a pandas data frame, but lucky for us, the movielens 100k data is one of their builtin data set. Let's load it and split off 20% for testing.

In [13]:
data = Dataset.load_builtin('ml-100k')

trainset, testset = train_test_split(data, test_size=.2, random_state=42)

Dataset ml-100k could not be found. Do you want to download it? [Y/n] Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /Users/lazaro/.surprise_data/ml-100k


We don't have to worry about any data transformation as Surprise takes care of this. This is why we can initialize and fit the data to the SVD algorithm right away. 

In [14]:
svd_algo = SVD(random_state = 42)
svd_algo.fit(trainset)

The `test` function will create the predictions for our test set. It returns a list with a user ID, item ID, the actual rating, and the estimated rating.

In [15]:
predictions_svd = svd_algo.test(testset)
predictions_svd[:10]

[Prediction(uid='907', iid='143', r_ui=5.0, est=4.611536250297749, details={'was_impossible': False}),
 Prediction(uid='371', iid='210', r_ui=4.0, est=4.37114223755087, details={'was_impossible': False}),
 Prediction(uid='218', iid='42', r_ui=4.0, est=3.738743910804353, details={'was_impossible': False}),
 Prediction(uid='829', iid='170', r_ui=4.0, est=3.8854605526859616, details={'was_impossible': False}),
 Prediction(uid='733', iid='277', r_ui=1.0, est=2.951243324162181, details={'was_impossible': False}),
 Prediction(uid='363', iid='1512', r_ui=1.0, est=3.1175626582710105, details={'was_impossible': False}),
 Prediction(uid='193', iid='487', r_ui=5.0, est=3.5842777082799806, details={'was_impossible': False}),
 Prediction(uid='808', iid='313', r_ui=5.0, est=4.694317679756497, details={'was_impossible': False}),
 Prediction(uid='557', iid='682', r_ui=2.0, est=3.188629840327263, details={'was_impossible': False}),
 Prediction(uid='774', iid='196', r_ui=3.0, est=2.438638505544496, deta

From manual inspection, you can see that a couple of predictions are a bit off while others are very close to the ground truth. Let's put a number to the predictive performance. As we are dealing with explicit data, we can use the `Root Mean Squared Error` (RMSE) to determine the predictive performance. With this metric, a lower value indicates a higher accuracy.

In [16]:
accuracy.rmse(predictions_svd)

RMSE: 0.9352


0.935171451026933

For better visibility, I copy/pasted the results of the first 3 users into this table:

| UserId   | SVD                      | NMF                    |
|:--------:|:-------------------------|:-----------------------|
| 907      |Juror, The (1996)         |Judge Dredd (1995)      |
|          |Johnny Mnemonic (1995)    |Larger Than Life (1996) |
|          |Courage Under Fire (1996) | Juror, The (1996)      |
|          |Flirting With Disaster (1996)| Johnny Mnemonic (1995)|
|          |Leaving Las Vegas (1995)  | Courage Under Fire (1996)|
|          |Net, The (1995)           | Net, The (1995)      |
|          |Judge Dredd (1995)        | Flirting With Disaster (1996)      |
|          |Brothers McMullen, The (1995)| Santa Clause, The (1994)   |
| 371      |Wild Bill (1995)          |  Kids (1995)     |
|          |Hate (Haine, La) (1995)   |  Wild Bill (1995)     |
|          |Nine Months (1995)        |  Hate (Haine, La) (1995)     |
|          |Forget Paris (1995)       |  Forget Paris (1995)     |
|          |Kids (1995)               |  Georgia (1995)     |
|          |Georgia (1995)            |  Nine Months (1995)     |
|          |Carlito's Way (1993)      |   Carlito's Way (1993)    |
| 218      |White Man's Burden (1995) |  Dracula: Dead and Loving It (1995) |
|          |Dracula: Dead and Loving It (1995)| White Man's Burden (1995)          |
|          |Dead Presidents (1995)    |    Devil in a Blue Dress (1995)          |
|          |Devil in a Blue Dress (1995)|  Dead Presidents (1995)          |
|          |Striptease (1996)         |    Seven (a.k.a. Se7en) (1995)          |
|          |Seven (a.k.a. Se7en) (1995)|    Striptease (1996)         |
|          |True Crime (1996)          |    True Crime (1996)         |


In [17]:
# Initialize and train SVD
svd_algo = SVD(n_factors=100, n_epochs=20, lr_all=0.005, reg_all=0.02)
svd_algo.fit(trainset)

# Generate predictions
predictions = svd_algo.test(testset)

# Get top N recommendations 
def get_top_n(predictions, n=10):
    top_n = defaultdict(list)
    for uid, iid, true_r, est, _ in predictions:
        top_n[uid].append((iid, est))
    
    # Sort and get top N
    for uid, user_ratings in top_n.items():
        user_ratings.sort(key=lambda x: x[1], reverse=True)
        top_n[uid] = user_ratings[:n]
    return top_n

# Get top 10 recommendations
recommendations = get_top_n(predictions, n=10)

In [18]:
print(recommendations[str(196)])  # Example for user with ID 196

[('306', 4.07108602283768), ('173', 3.9653503124287313), ('153', 3.9516374297133092), ('116', 3.9255127477866347), ('70', 3.507067685357015), ('762', 3.439644112774425), ('286', 3.422374228777213)]


In [22]:
def get_movie_recommendations(input_movies, movie_df, svd_algo, n_recommendations=10):
    """
    Get recommendations based on a list of input movies
    
    Args:
        input_movies (list): List of movie titles
        movie_df (pd.DataFrame): DataFrame containing movie information
        svd_algo: Trained SVD algorithm
        n_recommendations (int): Number of recommendations to return
    """
    # Find movie IDs for input movies
    input_movie_ids = []
    for movie in input_movies:
        movie_id = movie_df[movie_df['title'].str.contains(movie, case=False)]['movieId'].values
        if len(movie_id) > 0:
            input_movie_ids.append(movie_id[0])
    
    # Generate predictions for all movies
    all_movies = movie_df['movieId'].values
    predictions = []
    
    # Use a dummy user ID for predictions
    dummy_user = 'dummy_user'
    
    for movie_id in all_movies:
        pred = svd_algo.predict(dummy_user, movie_id)
        predictions.append((movie_id, pred.est))
    
    # Sort predictions and filter out input movies
    predictions.sort(key=lambda x: x[1], reverse=True)
    recommendations = [p for p in predictions if p[0] not in input_movie_ids][:n_recommendations]
    
    # Get movie titles
    recommended_movies = []
    for movie_id, score in recommendations:
        title = movie_df[movie_df['movieId'] == movie_id]['title'].values[0]
        recommended_movies.append((title, score))
    
    return recommended_movies

# Example usage:
input_movies = ["Nightmare on Elm Street", "Friday the 13th", "Halloween"]
recommendations = get_movie_recommendations(input_movies, movie_df, svd_algo)

# Print recommendations
for movie, score in recommendations:
    print(f"{movie}: {score:.2f}")

Toy Story (1995): 3.53
Jumanji (1995): 3.53
Grumpier Old Men (1995): 3.53
Waiting to Exhale (1995): 3.53
Father of the Bride Part II (1995): 3.53
Heat (1995): 3.53
Sabrina (1995): 3.53
Tom and Huck (1995): 3.53
Sudden Death (1995): 3.53
GoldenEye (1995): 3.53


In [25]:
# Load movies with genres
movie_df = pd.read_csv('cache/recommendation/movies.csv')
# Split genres string into a list
movie_df['genres'] = movie_df['title'].str.extract(r'\((.*?)\)$')
movie_df['year'] = movie_df['genres'].fillna('').astype(str)
movie_df['genres'] = movie_df['title'].str.extract(r'\((.*?)\)')[0].str.split('|')

def get_movie_recommendations(input_movies, movie_df, svd_algo, n_recommendations=10):
    """
    Get personalized recommendations based on input movies using genre similarity
    """
    # Find movie IDs and genres for input movies
    input_movie_data = []
    for movie in input_movies:
        matches = movie_df[movie_df['title'].str.contains(movie, case=False)]
        if not matches.empty:
            movie_id = matches.iloc[0]['movieId']
            genres = matches.iloc[0]['genres']
            year = matches.iloc[0]['year']
            input_movie_data.append((movie_id, genres, year))
    
    # Collect all genres from input movies
    all_input_genres = []
    input_years = []
    for _, genres, year in input_movie_data:
        if isinstance(genres, list):
            all_input_genres.extend(genres)
        if year:
            input_years.append(year)
    
    # Count genre frequencies
    genre_counts = pd.Series(all_input_genres).value_counts()
    
    # Generate predictions for all movies
    all_movies = movie_df['movieId'].values
    predictions = []
    dummy_user = 'dummy_user'
    
    for movie_id in all_movies:
        if movie_id not in [x[0] for x in input_movie_data]:  # Skip input movies
            movie_info = movie_df[movie_df['movieId'] == movie_id].iloc[0]
            pred = svd_algo.predict(dummy_user, movie_id)
            
            # Calculate genre similarity score
            genre_score = 0
            if isinstance(movie_info['genres'], list):
                for genre in movie_info['genres']:
                    if genre in genre_counts:
                        genre_score += genre_counts[genre]
            
            # Adjust score based on both SVD prediction and genre similarity
            adjusted_score = pred.est * (1 + genre_score/len(all_input_genres))
            
            # Add year similarity bonus if applicable
            if movie_info['year'] in input_years:
                adjusted_score *= 1.2
            
            predictions.append((movie_id, adjusted_score))
    
    # Sort and get top recommendations
    predictions.sort(key=lambda x: x[1], reverse=True)
    recommendations = predictions[:n_recommendations]
    
    # Get movie titles and scores
    recommended_movies = []
    for movie_id, score in recommendations:
        title = movie_df[movie_df['movieId'] == movie_id]['title'].values[0]
        recommended_movies.append((title, score))
    
    return recommended_movies

# Test with different genres
print("Recommendations for horror movies:")
horror_movies = ["Nightmare on Elm Street", "Friday the 13th", "Halloween"]
recommendations = get_movie_recommendations(horror_movies, movie_df, svd_algo)
print("Input movies genres:", [movie_df[movie_df['title'].str.contains(m, case=False)].iloc[0]['genres'] for m in horror_movies])
for movie, score in recommendations:
    print(f"{movie}: {score:.2f}")

print("\nRecommendations for Disney movies:")
disney_movies = ["The Lion King", "Beauty and the Beast", "Pocahontas"]
recommendations = get_movie_recommendations(disney_movies, movie_df, svd_algo)
print("Input movies genres:", [movie_df[movie_df['title'].str.contains(m, case=False)].iloc[0]['genres'] for m in disney_movies])
for movie, score in recommendations:
    print(f"{movie}: {score:.2f}")

Recommendations for horror movies:
Input movies genres: [["Nightmare on Elm Street Part 7: Freddy's Finale, A"], ['1980'], ['Halloween 6: The Curse of Michael Myers']]
Fog, The (1980): 5.65
Howling, The (1980): 5.65
Private Benjamin (1980): 5.65
Star Wars: Episode V - The Empire Strikes Back (1980): 5.65
Blues Brothers, The (1980): 5.65
Raging Bull (1980): 5.65
Shining, The (1980): 5.65
Somewhere in Time (1980): 5.65
Ordinary People (1980): 5.65
Prom Night (1980): 5.65

Recommendations for Disney movies:


IndexError: single positional indexer is out-of-bounds